# Inheritance and Polymorphism

**Inheritance** lets a class (child) reuse code from another class (parent).

**Polymorphism** means different classes can respond to the same method call in different ways.

These are two of the four pillars of OOP (along with Encapsulation and Abstraction).

By the end of this notebook you will be able to:
- Create child classes that extend parent classes
- Override methods
- Use `super()` correctly
- Understand multiple inheritance and MRO
- Use `isinstance()` and `issubclass()`
- Define abstract classes with `abc`

## 1. Basic Inheritance

In [ ]:
class Animal:
    """Base class for all animals."""
    
    def __init__(self, name, species):
        self.name = name
        self.species = species
    
    def speak(self):
        return f'{self.name} makes a sound'
    
    def describe(self):
        return f'{self.name} is a {self.species}'
    
    def __str__(self):
        return f'{self.species}({self.name})'


class Dog(Animal):  # Dog inherits from Animal
    def __init__(self, name, breed):
        super().__init__(name, 'Dog')  # call parent __init__
        self.breed = breed
    
    def speak(self):  # override parent method
        return f'{self.name} says: Woof!'
    
    def fetch(self, item):
        return f'{self.name} fetches the {item}'


class Cat(Animal):
    def __init__(self, name, indoor=True):
        super().__init__(name, 'Cat')
        self.indoor = indoor
    
    def speak(self):
        return f'{self.name} says: Meow!'


dog = Dog('Rex', 'Labrador')
cat = Cat('Whiskers')

print(dog.speak())     # Dog-specific
print(cat.speak())     # Cat-specific
print(dog.describe())  # inherited from Animal
print(dog.fetch('ball'))  # Dog-only
print(dog)
print(cat)

## 2. `super()` — Calling the Parent Class

`super()` returns a proxy to the **parent class**, letting you call its methods.

In [ ]:
class Vehicle:
    def __init__(self, make, model, year):
        self.make = make
        self.model = model
        self.year = year
    
    def describe(self):
        return f'{self.year} {self.make} {self.model}'
    
    def start(self):
        return f'{self.model} engine started'


class ElectricVehicle(Vehicle):
    def __init__(self, make, model, year, battery_kwh):
        super().__init__(make, model, year)  # initialize Vehicle part
        self.battery_kwh = battery_kwh  # add EV-specific attribute
    
    def describe(self):
        base = super().describe()  # reuse parent's describe
        return f'{base} (Electric, {self.battery_kwh}kWh battery)'
    
    def start(self):
        return f'{self.model} motor silently engaged'


ev = ElectricVehicle('Tesla', 'Model 3', 2024, 82)
print(ev.describe())
print(ev.start())
print(ev.year)  # inherited attribute

## 3. Polymorphism — Same Interface, Different Behavior

Each class can implement the same method name differently. Callers don't need to know which specific type they have.

In [ ]:
class Shape:
    def area(self):
        raise NotImplementedError('Subclasses must implement area()')
    
    def __str__(self):
        return f'{self.__class__.__name__} with area {self.area():.2f}'

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    
    def area(self):
        import math
        return math.pi * self.radius ** 2

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height
    
    def area(self):
        return self.width * self.height

class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height
    
    def area(self):
        return 0.5 * self.base * self.height


# Polymorphism: the same code works for any Shape
shapes = [Circle(5), Rectangle(4, 6), Triangle(3, 8)]

total_area = 0
for shape in shapes:
    print(shape)  # each responds differently
    total_area += shape.area()

print(f'\nTotal area: {total_area:.2f}')

## 4. `isinstance()` and `issubclass()`

In [ ]:
dog = Dog('Buddy', 'Poodle')
cat = Cat('Felix')

# isinstance — check if object is an instance of a class (or its subclasses)
print(isinstance(dog, Dog))     # True
print(isinstance(dog, Animal))  # True  — Dog inherits from Animal
print(isinstance(dog, Cat))     # False

# issubclass — check class hierarchy
print(issubclass(Dog, Animal))  # True
print(issubclass(Cat, Animal))  # True
print(issubclass(Dog, Cat))     # False

# Useful for type-safe code
def make_sound(animal):
    if not isinstance(animal, Animal):
        raise TypeError('Expected an Animal')
    return animal.speak()

print(make_sound(dog))
print(make_sound(cat))

## 5. Abstract Classes with `abc`

An **abstract class** cannot be instantiated directly. It defines a contract — subclasses **must** implement certain methods.

In [ ]:
from abc import ABC, abstractmethod

class Payment(ABC):
    """Abstract base class for payment methods."""
    
    @abstractmethod
    def process(self, amount):
        """Process a payment of the given amount."""
        pass
    
    @abstractmethod
    def refund(self, amount):
        """Refund the given amount."""
        pass
    
    def receipt(self, amount):
        """Concrete method available to all subclasses."""
        return f'Receipt: £{amount:.2f} processed via {self.__class__.__name__}'


class CreditCard(Payment):
    def __init__(self, card_number):
        self.card_number = f'****{card_number[-4:]}'
    
    def process(self, amount):
        print(f'Charging £{amount:.2f} to {self.card_number}')
    
    def refund(self, amount):
        print(f'Refunding £{amount:.2f} to {self.card_number}')


class PayPal(Payment):
    def __init__(self, email):
        self.email = email
    
    def process(self, amount):
        print(f'Charging £{amount:.2f} via PayPal ({self.email})')
    
    def refund(self, amount):
        print(f'Refunding £{amount:.2f} to PayPal ({self.email})')


# Cannot instantiate abstract class
try:
    p = Payment()
except TypeError as e:
    print(f'Error: {e}')

card = CreditCard('1234567890123456')
card.process(49.99)
print(card.receipt(49.99))

paypal = PayPal('alice@example.com')
paypal.process(19.99)

## 6. Multiple Inheritance and MRO

Python supports inheriting from **multiple** parent classes. The **Method Resolution Order (MRO)** determines which parent's method is called.

In [ ]:
class Flyable:
    def move(self):
        return 'Flying'

class Swimmable:
    def move(self):
        return 'Swimming'

class Duck(Flyable, Swimmable):
    """Duck can both fly and swim."""
    pass

class Penguin(Swimmable):
    pass

d = Duck()
print(d.move())  # 'Flying' — Flyable comes first in MRO

# Check MRO
print(Duck.__mro__)

# Mixin pattern — add capabilities via inheritance
class LogMixin:
    def log(self, message):
        print(f'[{self.__class__.__name__}] {message}')

class SmartDog(Dog, LogMixin):
    def fetch(self, item):
        self.log(f'Fetching {item}')
        return super().fetch(item)

sd = SmartDog('Max', 'Border Collie')
print(sd.fetch('ball'))

## Common Mistakes

### Mistake 1: Not Calling `super().__init__()`

In [ ]:
class Parent:
    def __init__(self):
        self.parent_attr = 'from parent'

# BAD — parent attributes never set!
class BadChild(Parent):
    def __init__(self):
        self.child_attr = 'from child'
        # forgot: super().__init__()

# GOOD
class GoodChild(Parent):
    def __init__(self):
        super().__init__()  # initialize parent first
        self.child_attr = 'from child'

bad = BadChild()
try:
    print(bad.parent_attr)  # AttributeError!
except AttributeError as e:
    print(f'Bad child error: {e}')

good = GoodChild()
print(f'Good child has: {good.parent_attr} and {good.child_attr}')

## Exercises

### Exercise 1 (Easy): Vehicle Hierarchy
Create `Car` and `Motorcycle` classes inheriting from `Vehicle`. Each should have a `describe()` method.

In [ ]:
# Your solution here


In [ ]:
# Solution
class Vehicle:
    def __init__(self, make, model, year):
        self.make = make
        self.model = model
        self.year = year

class Car(Vehicle):
    def __init__(self, make, model, year, doors=4):
        super().__init__(make, model, year)
        self.doors = doors
    
    def describe(self):
        return f'{self.year} {self.make} {self.model} ({self.doors}-door car)'

class Motorcycle(Vehicle):
    def __init__(self, make, model, year, style='standard'):
        super().__init__(make, model, year)
        self.style = style
    
    def describe(self):
        return f'{self.year} {self.make} {self.model} ({self.style} motorcycle)'

car = Car('Toyota', 'Camry', 2023)
moto = Motorcycle('Honda', 'CBR', 2023, 'sport')
for v in [car, moto]:
    print(v.describe())

### Exercise 2 (Hard): Employee Hierarchy with Abstract Base

In [ ]:
# Create:
# - Abstract Employee with name, id, and abstract calculate_pay()
# - HourlyEmployee(hours, rate)
# - SalariedEmployee(annual_salary)
# - Commission-based employee (base + commission_rate + sales)
# Your solution here


In [ ]:
# Solution
from abc import ABC, abstractmethod

class Employee(ABC):
    def __init__(self, name, emp_id):
        self.name = name
        self.emp_id = emp_id
    
    @abstractmethod
    def calculate_pay(self):
        pass
    
    def payslip(self):
        print(f'ID:{self.emp_id} | {self.name:15} | Pay: £{self.calculate_pay():,.2f}')

class HourlyEmployee(Employee):
    def __init__(self, name, emp_id, hours, rate):
        super().__init__(name, emp_id)
        self.hours = hours
        self.rate = rate
    
    def calculate_pay(self):
        return self.hours * self.rate

class SalariedEmployee(Employee):
    def __init__(self, name, emp_id, annual_salary):
        super().__init__(name, emp_id)
        self.annual_salary = annual_salary
    
    def calculate_pay(self):
        return self.annual_salary / 12  # monthly pay

class CommissionEmployee(Employee):
    def __init__(self, name, emp_id, base, commission_rate, sales):
        super().__init__(name, emp_id)
        self.base = base
        self.commission_rate = commission_rate
        self.sales = sales
    
    def calculate_pay(self):
        return self.base + (self.sales * self.commission_rate)

staff = [
    HourlyEmployee('Alice', 'E001', 160, 20),
    SalariedEmployee('Bob', 'E002', 60000),
    CommissionEmployee('Carol', 'E003', 2000, 0.1, 15000),
]

print('=== Monthly Payroll ===')
for emp in staff:
    emp.payslip()